In [ ]:
import numpy as np
import pandas as pd
import h5py
import yaml
import h5flow

from sklearn.cluster import DBSCAN
from scipy.stats import chisquare

from draw_utils import *
from cluster_utils import *
from mapping_utils import *

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex


# Input files

In [ ]:
# flow file
f_name = '/global/cfs/cdirs/dune/www/data/2x2/people/mkramer/ndlar_pileup/v2/MicroProdN1p1_NDLAr_1E18_RHC.larnd.nu.0000002.20250422T145519Z.v2.FLOW.hdf5'
data = h5flow.data.H5FlowDataManager(f_name, 'r')


In [ ]:
# FSD LUT
vis_lut = np.load('/global/cfs/cdirs/dune/www/data/2x2/simulation/larndsim_data/light_LUT/lightLUT_FSD_250123_time_norm.npz')['arr']['vis']
lut_vox_div = vis_lut.shape[:-1]


In [ ]:
light_channel_map_file = 'LightEventGeneratorMC.yaml'

with open(light_channel_map_file) as fi:
    yfi = yaml.full_load(fi)
    light_channel_map = yfi['params']['channel_map']
    n_sipms_per_module = yfi['params']['n_sipms_per_module']
    n_adc_channels = yfi['params']['n_channels']

light_channel_map = np.array(light_channel_map)

# Useful constants

In [ ]:
mod_bounds = data['geometry_info'].attrs['module_RO_bounds']
det_bounds = data['geometry_info'].attrs['lar_detector_bounds']
tpc_center, anodes, cathodes, cages = draw_tpc(geo_version="default", mod_bounds=mod_bounds, det_bounds=det_bounds)

NTPCS = 35*2
NOPCHAN = 120


# Main loop

In [ ]:

for ievt in [0]:
    
    PromptHits_ev = data["charge/events", "charge/calib_prompt_hits", ievt]
    LightSiPMHits_ev = data['light/events', 'light/sipm_hits', ievt]


    all_charge_clusters = []
    all_charge_clusters_light_timestamps = []
    
    # Loop over TPCs
    
    print(f'Event {ievt}')
    
    for itpc in range(NTPCS):
        
        print(f'TPC {itpc}')

        # Charge
        
        PromptHits_tpc = PromptHits_ev[io_group_to_tpc(PromptHits_ev['io_group']) == itpc] 
        
        if PromptHits_tpc.shape[0] == 0:
            # no charge hits
            continue
        
        points = np.array([PromptHits_tpc['x'], PromptHits_tpc['y'], PromptHits_tpc['z']]).T

        # Apply DBSCAN
        pixel_pitch = 0.372
        eps = pixel_pitch * 5
        min_samples = 10
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(points)

        unique_labels = sorted(set(labels))

        charge_clusters = [PromptHits_tpc[labels == l] for l in unique_labels if l != -1]

        print(f'Found {len(charge_clusters)} charge clusters')
        # fig1 = plot_clusters([points[labels == l] for l in unique_labels if l != -1], title='DBScan step')
        # fig1.show()

        if len(charge_clusters) == 0:
            continue

        all_charge_clusters += charge_clusters
        # Light

        # list storing the timestamp obtained from light matching for each charge cluster 
        charge_clusters_light_timestamps = [-1 for _ in charge_clusters] 

        # first select sipm hits in the TPC
        LightSiPMHits_tpc = LightSiPMHits_ev[adc_chan_to_tpc(LightSiPMHits_ev['adc'], LightSiPMHits_ev['chan']) == itpc] 

        # Convert (adc, chan) to LUT channel index
        ilut = adc_chan_to_ilut(LightSiPMHits_tpc['adc'], LightSiPMHits_tpc['chan'], light_channel_map, NOPCHAN)

        if LightSiPMHits_tpc.shape[0] == 0:
            # no light hits
            print('No sipm hits')
            all_charge_clusters_light_timestamps += charge_clusters_light_timestamps
            continue

        # Cluster light hits by their timestamps
        light_cluster_indices = cluster_neighboring_timestamps(LightSiPMHits_tpc['sample_idx'])
        
        print(f'Found {len(light_cluster_indices)} light clusters in TPC {itpc}')
        
        # Which light cluster corresponds to which charge cluster?
        
        matched_light_clusters = []
        
        # simple cases
        
        if len(light_cluster_indices) == 0:
            print(f'Skipping no light cluster, despite {LightSiPMHits_tpc.shape[0]} sipm hits')
            print(LightSiPMHits_tpc['sample_idx'].flatten())
            all_charge_clusters_light_timestamps += charge_clusters_light_timestamps
            continue
       
        if len(light_cluster_indices) == 1:
            charge_clusters_light_timestamps = [np.mean(LightSiPMHits_tpc['sample_idx'][light_cluster_indices[0]])] * len(charge_clusters)
            all_charge_clusters_light_timestamps += charge_clusters_light_timestamps
            continue
        
        # Use LUT for more complicated cases
        
        vis_per_opchan_clusters = []
        for cc in charge_clusters:
            
            vis_per_opchan = np.zeros(NOPCHAN)
            
            for c in cc:
                # LUT expects (z,y,x)
                c_lut = (c['z'], c['y'], c['x'])
                vox = get_voxel(c_lut, itpc, lut_vox_div)
                vis_per_opchan += vis_lut[vox] * c['Q'] # multiply the charge/edep

            vis_per_opchan_clusters.append(vis_per_opchan)
            
        # get vis from all possible charge cluster combinations
        # is this necessary? Extremely intensive computation
        # cs = all_array_sums(vis_per_opchan_clusters)
        cs = vis_per_opchan_clusters
        print(f'{len(cs)} possible combinations from {len(charge_clusters)} charge clusters')
        
        ls = []
            
        for ilc in range(len(light_cluster_indices)):
            # dataframe with hit timestamps, LUT index,
            df = pd.DataFrame({
                'sample_idx': LightSiPMHits_tpc['sample_idx'][light_cluster_indices[ilc]],
                'ilut':       ilut[light_cluster_indices[ilc]],
                'sum_spline': (LightSiPMHits_tpc['sum_spline'][light_cluster_indices[ilc]]),
                'max_spline': (LightSiPMHits_tpc['max_spline'][light_cluster_indices[ilc]]),
            })
            # print(df)
            # just in case two hits with the close timestamps on the same sipm
            agg_df = df.groupby(['ilut', 'sample_idx'], as_index=False)['sum_spline'].sum()
            # print(agg_df)
            agg_df['norm'] = agg_df['sum_spline'] / agg_df['sum_spline'].sum()
            ls.append(agg_df)
    
        
        for ic in range(len(cs)):

            # for each charge cluster, loop over light clusters
            # and compare dist. of amplitudes on SiPMs via chi2

            # expected light dist. from LUT
            # this array is indexed with LUT indices
            exp = (np.array(cs[ic])) 
            exp /= exp.sum() 
            
            # fig, ax1 = plt.subplots()
            
            # ax1.plot(exp, 'b-', label='exp')
            # ax1.set_xlabel('SiPM')
            # ax1.set_ylabel('vis', color='b')
            # ax1.tick_params(axis='y', labelcolor='b')

            # # Second axis
            # ax2 = ax1.twinx()

            chi2s = []
            
            for il in range(len(ls)):
                
                # observed light dist.
                # this array is indexed with LUT indices
                obs = np.zeros(NOPCHAN)
                
                obs[ls[il]['ilut']] = ls[il]['norm']
                
                obs *= (exp.sum() / obs.sum())
                chi2 = chisquare(f_obs=obs, f_exp=exp)[0]
                chi2s.append(chi2)
                # ax2.plot(obs, label=f'obs, chi2={chi2}')

            # ax2.set_ylabel('Hit', color='r')
            # ax2.tick_params(axis='y', labelcolor='r')
            # plt.title(f'TPC {itpc}')
            # plt.legend()

            min_chi2_light_cluster = chi2s.index(min(chi2s))
            charge_clusters_light_timestamps[ic] = np.mean(ls[min_chi2_light_cluster]['sample_idx'])
        
        all_charge_clusters_light_timestamps += charge_clusters_light_timestamps

            

In [ ]:
all_charge_clusters_xyz = []
for c in all_charge_clusters:
    l = []
    for cc in c:
        l.append((cc['x'], cc['y'], cc['z']))
    all_charge_clusters_xyz.append(l)

merged_clusters = merge_clusters(all_charge_clusters_xyz, all_charge_clusters_light_timestamps, 3)

In [ ]:
fig = plot_clusters(merged_clusters, tpc_center, anodes, cathodes, cages)

fig.show()

# Truth backtracking

In [ ]:
ievt = 1

PromptHits_ev = data["charge/events", 
                     "charge/calib_prompt_hits", 
                     ievt]
Segs_PromptHits = data["charge/events",
                       "charge/calib_prompt_hits",
                       "charge/packets", 
                       "mc_truth/segments", 
                       ievt]

unique_vertices = np.unique(Segs_PromptHits[0,:,0,0].data['vertex_id'])

In [ ]:
true_clusters = [np.array(
    [PromptHits_ev.data['x'].flatten()[Segs_PromptHits[0,:,0,0].data['vertex_id'] == vertex_id], 
    PromptHits_ev.data['y'].flatten()[Segs_PromptHits[0,:,0,0].data['vertex_id'] == vertex_id],
    PromptHits_ev.data['z'].flatten()[Segs_PromptHits[0,:,0,0].data['vertex_id'] == vertex_id]]).T 
                 for vertex_id in unique_vertices]

In [ ]:
fig = plot_clusters(true_clusters, tpc_center, anodes, cathodes, cages, ncol=10)
fig.write_html(f"ndlar_event_{ievt}_truth_test.html", include_plotlyjs='cdn')


In [ ]:
def round_tuples(arr, decimals=6):
    return set(map(tuple, np.round(arr, decimals)))


In [ ]:
results = np.zeros((len(merged_clusters), len(true_clusters)))
for ic in range(len(merged_clusters)):
    # how many hits in this cluster are in the truth cluster?
#     print(f'Cluster {ic}')
    for itc in range(len(true_clusters)):
        reco = round_tuples(merged_clusters[ic])
        truth = round_tuples(true_clusters[itc])
        common_count = len(reco & truth)
    
        results[ic, itc] = common_count/len(merged_clusters[ic])


In [ ]:
np.round(results, 3)